In [1]:
!pip install mrob

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for mrob from https://files.pythonhosted.org/packages/03/6c/8f59ec9562733de97ba78adae7577bdff2457bd9726cd095114a7ba80f4e/mrob-0.0.17-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 712.6/712.6 kB 648.3 kB/s eta 0:00:00a 0:00:01


In [2]:
import mrob
import numpy as np

# Graph SLAM, using a library
in this seminar, we will show some of the functionalities of using a library for solving graphSLAM. [MROB: Mobile Robotics library](https://github.com/prime-slam/mrob) is a library for general use of perception techniques: FGraphs, Rigid Body Transformation and Point Cloud alignment.

We will show two basic problems in 2D and discuss the 

Other interesting libraries to explore are g2o (Kumerle'2011) and GTSAM (Dellaert'2011).

to install, simply ```pip install mrob```

## 1 Creating a Graph
We will start by creating a graph, and then add a node. You can get familiar by using help or from the examples in mrob (see github python_examples)

In [21]:
graph = mrob.FGraph()

In [22]:
with mrob.ostream_redirect(stdout=True, stderr=True):
      graph.print()

Status of graph:  Nodes = 0, Factors = 0, Eigen Factors = 0


## 2. Add the first Node $x_0$
We will add the first node to the Fgraph. Create a random initial state ($\sigma = 0.1 $)and add it to the graph. For that, use the function add_node_pose_2d().

Print your graph in simplified mode and in complete form.

In [5]:
?graph.add_node_pose_2d

Docstring:
add_node_pose_2d(self: mrob.pybind.FGraph, x: numpy.ndarray[numpy.float64[3, 1]], mode: mrob.pybind.FGraph.nodeMode = <FGraph.nodeMode.NODE_STANDARD: 0>) -> int

 - arguments, initial estimate (np.zeros(3)
output, node id, for later usage
Type:      method

In [23]:
r_state = np.random.randn(3) * 0.1
r_state

array([-0.06707643, -0.10201154,  0.13400142])

In [24]:
n1 = graph.add_node_pose_2d(r_state)

In [25]:
?graph.print
with mrob.ostream_redirect(stdout=True, stderr=True):
      graph.print(True)

Status of graph:  Nodes = 1, Factors = 0, Eigen Factors = 0
Printing NodePose2d: 0, state = 
-0.0670764
 -0.102012
  0.134001


Docstring:
print(self: mrob.pybind.FGraph, completePrint: bool = False) -> None

By default False: does not print all the information on the Fgraph
Type:      method

## 3. Add a factor to $x_0$
Now that we have a node in the graph, we want to add the first observation. In this case it will be an anchor factor, assuming we are observing that the node is at $[0,0,0]$ with information $\Sigma_{x_0}= 10^6 I$ 

In [26]:
?graph.add_factor_1pose_2d

init_cov = 10**6 * np.identity(3)

graph.add_factor_1pose_2d(np.zeros(3), n1, init_cov)
with mrob.ostream_redirect(stdout=True, stderr=True):
      graph.print(True)

Status of graph:  Nodes = 1, Factors = 1, Eigen Factors = 0
Printing NodePose2d: 0, state = 
-0.0670764
 -0.102012
  0.134001
Printing Factor: 0, obs= 
0
0
0
 Residuals= 
 5.8368e-302
 2.4933e-306
3.05333e-321 
and Information matrix
1e+06     0     0
    0 1e+06     0
    0     0 1e+06
 Calculated Jacobian = 
0 0 0
0 0 0
0 0 0
 Chi2 error = 0 and neighbour Nodes 1


Docstring: add_factor_1pose_2d(self: mrob.pybind.FGraph, arg0: numpy.ndarray[numpy.float64[3, 1]], arg1: int, arg2: numpy.ndarray[numpy.float64[3, 3]]) -> None
Type:      method

## 4. Analize the current error in the graph
For this, use the function chi2, which evaluates the problem at the current point and calculates the current value of the residuals.

You can also get the current state estimate by using the function get_estimated_state(). Print its current value.

In [29]:
cur_state = graph.get_estimated_state()
cur_state

[array([[-0.06707643],
        [-0.10201154],
        [ 0.13400142]])]

In [33]:
graph.chi2()

16430.99101201376

## 5. Solve
We will use the Gauss Newton routine (mrob.GN) with one iteration. For that, call the function solve() and reculate the current estimate and the error.

In [ ]:
?graph.solve

In [34]:
graph.solve(mrob.GN)

1

In [35]:
with mrob.ostream_redirect(stdout=True, stderr=True):
    
    graph.print(True)

Status of graph:  Nodes = 1, Factors = 1, Eigen Factors = 0
Printing NodePose2d: 0, state = 
0
0
0
Printing Factor: 0, obs= 
0
0
0
 Residuals= 
-0.0670764
 -0.102012
  0.134001 
and Information matrix
1e+06     0     0
    0 1e+06     0
    0     0 1e+06
 Calculated Jacobian = 
1 0 0
0 1 0
0 0 1
 Chi2 error = 16431 and neighbour Nodes 1
